In [1]:
# I didn't want to have to fuck with the import shit so I just did this to save time lmao
import os
import json
from typing import List, Dict, Any
import re

def parse_book(path: str) -> List[Dict[str, Any]]:
    """
    Parses the book into chapters and sections.
    Save the parsed book to a JSON file in the output directory.

    Args:
        path: Path to the book file.
    """

    # Check if the book file exists
    if not os.path.exists(path):
        raise FileNotFoundError(f"book file not found at {path}")

    print(f"Parsing book {path}...")

    # Read the book file
    with open(path, "r", encoding="utf-8") as f:
        file_content = f.read()

    # PARSE THE BOOK BY CHAPTERS AND SECTIONS
    parsed_book = []
    ch_blocks = re.split(r"^#CHAPTER\s*", file_content, flags=re.MULTILINE)
    for ch_idx, ch_block in enumerate(ch_blocks):
        if not ch_block:
            continue

        # Separate chapter title from text
        lines = ch_block.splitlines()
        ch_title = lines[0].strip()
        ch_text = "\n".join(lines[1:])

        # Split sections within the chapter
        sections = []
        sec_blocks = re.split(r"^#SECTION\s*", ch_text, flags=re.MULTILINE)

        # Case 1: No #SECTION in chapter. Treat entire chapter text as a single, untibtled section
        if len(sec_blocks) == 1:
            chapter_description = sec_blocks[0].strip()
            if not chapter_description:
                chapter_description = "<No text in section>"

            sections.append(
                {
                    "section_idx": 0,
                    "section_title": "<Chapter description>",
                    "section_text": chapter_description,
                }
            )
        else:
            # Case 2: There are sections. The first block may be the chapter description
            running_idx = 0
            chapter_description = sec_blocks[0].strip()
            if chapter_description:
                sections.append(
                    {
                        "section_idx": running_idx,
                        "section_title": "<Chapter description>",
                        "section_text": chapter_description,
                    }
                )
                running_idx += 1

            # Parse each real section: first non-empty line is the title, rest is content
            for sec_block in sec_blocks[1:]:
                block = sec_block.strip()
                if not block:
                    continue

                sec_lines = sec_block.splitlines()
                sec_title = sec_lines[0].strip()
                sec_text = "\n".join(sec_lines[1:]).strip()
                if not sec_text:
                    sec_text = "<No text in section>"

                sections.append(
                    {
                        "section_idx": running_idx,
                        "section_title": sec_title,
                        "section_text": sec_text,
                    }
                )
                running_idx += 1

        # Add the chapter to the parsed book
        parsed_book.append(
            {
                "chapter_idx": ch_idx,
                "chapter_title": ch_title,
                "sections": sections,
            }
        )

    return parsed_book

In [2]:
# 1. parse document using the tagging/parse_books.py
parsed_book = parse_book("../books/《人紀傷寒論》.txt")

Parsing book ../books/《人紀傷寒論》.txt...


In [3]:
# Tokenize each document
import jieba

jieba.set_dictionary('dict.txt.big')

corpus = []
for chapter in parsed_book:
    for section in chapter["sections"]:
        corpus.append(((chapter["chapter_idx"], section["section_idx"]), section["section_text"]))

tokenized_corpus = [list(jieba.cut(doc)) for _, doc in corpus]

/Users/hughesh/temp/yQi/.venv/lib/python3.13/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from /Users/hughesh/temp/yQi/experiment2-bm25/dict.txt.big ...
Loading model from cache /var/folders/1x/857dmnt90l9_vghkr56p1f1w0000gp/T/jieba.u0f420bc0d45a1e7e0bbb1b8240110fe7.cache
Loading model cost 0.400 seconds.
Prefix dict has been built successfully.


In [8]:
len(corpus)

403

In [4]:
# Create BM25 index
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

In [5]:
queries = [
  "病人，男，20歲，亞裔，幾天前開始覺得不舒服，頭痛，身體發熱，怕吹風，有些流汗。",
  "病人，女，20歲，亞裔，自己覺得好像感冒了，頭頸疼，後背很緊的感覺，怕冷，怕風，沒有流汗。",
  "病人，女，20歲，亞裔，自己覺得好像感冒了，怕冷，身體沉重，說話會喘，有些乾嘔，肚子一些漲感，小便不易出。",
  "病人，女，30歲，亞裔，自己覺得好像感冒了，怕冷，頭痛，身體有些發熱，也感覺身體沉重，好幾天沒有大便了。",
  "病人，男，30歲，白人，小便不順暢，身體微微發熱，一直感到口渴。",
  "病人，男，30歲，白人，自己覺得好像感冒了，身體流汗後依然發熱，胸口下方有些悸動，頭暈，身體微微顫動。",
  "病人，女，35歲，亞裔，感冒好多天了，身體疼痛，全身關節也痛，躺在時很難自己轉動身體，拉肚子，有些小便失禁的情況。",
  "病人，女，40歲，白人，最近小便不順暢，有時候也大便困難，大便乾硬，身體偶然發熱，躺下來時會喘咳不舒服。",
  "病人，女，35歲，亞裔，前幾天感冒，嘔吐，現在覺得肚子很漲。",
  "病人，女，40歲，亞裔，脈搏很細弱，白天想睡覺卻睡不著，感覺心裡很煩，無法躺下來安心睡覺。",
  "病人，女，40歲，亞裔，脈搏很細弱也很沉，白天想睡覺卻睡不著，身體疼痛，手腳冷，關節疼痛。",
  "病人，男，50歲，白人，最近覺得口很渴，胸口疼痛，有氣往上逆的感覺，肚子餓卻不想吃東西，吃了就想吐。",
  "病人，女，50歲，亞裔，手腳非常冷，脈搏非常細微。",
  "病人，女，60歲，亞裔，連續幾天拉肚子，心裡越來越煩躁，按肚子覺得有些脹滿的感覺，肚子卻沒有很硬。"
]

expected_idxs = [
  (3, 16),
  (3, 35),
  (3, 45),
  (4, 15),
  (4, 31),
  (4, 44),
  (5, 56),
  (6, 64),
  (6, 71),
  (9, 23),
  (9, 25),
  (10, 1),
  (10, 28),
  (10, 52)]

In [6]:
# Run BM25 for given query
tokenized_queries = [list(jieba.cut(query)) for query in queries]

In [11]:
# Get top matches
# store the ranking
k = 200
query_rankings_k25 = []
for i, tokenized_query in enumerate(tokenized_queries):
    scores = bm25.get_scores(tokenized_query)
    ranked = sorted(zip(scores, corpus), reverse=True)

    query_rankings_k25.append((queries[i], [doc_idx_tuple for _, doc_idx_tuple in ranked[:k]]))
    
    # save to a file
    file_path = f'output/query_{i}.txt'
    with open(file_path, 'w') as f:
        f.write(f'query {i}\n')
        f.write(f'query: {queries[i]}\n')
        f.write('----------\n')
        for j, (score, doc_idx_tuple) in enumerate(ranked[:k]):
            f.write(f"rank: {j}\n")
            f.write(f"score: {score:.3f}\n")
            if doc_idx_tuple[0] == expected_idxs[i]:
                f.write(f"!!expected section!!\n")
            f.write(f"chapter_idx: {doc_idx_tuple[0][0]}\n")
            f.write(f"section_idx: {doc_idx_tuple[0][1]}\n")
            f.write(f"text: {doc_idx_tuple[1]}\n")
            f.write('=====\n')
        f.write('\n\n')

In [1]:
# Reranking using Cross Encoder
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "BAAI/bge-reranker-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=1024, ou

In [ ]:
pairs = [[query_rankings_k25[0][0], doc] for doc in query_rankings_k25[0][1]]
print(pairs)

In [ ]:
with torch.no_grad():
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
    scores = model(**inputs, return_dict=True).logits.view(-1, ).float()
    print(scores)